# Sprint 5 – Notebook 10: Feature Selection



In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif, mutual_info_classif, RFE
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

X_array, y_array = make_classification(n_samples=120, n_features=5, n_informative=3, n_redundant=0, random_state=42)
df = pd.DataFrame(X_array, columns=["Age_Score", "Income_Score", "Visit_Score", "Random_Score", "Engagement_Score"])
df["Constant"] = 1
df["Age_Copy"] = df["Age_Score"]
df["Purchased"] = y_array
X = df.drop(columns="Purchased")
y = df["Purchased"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
print("Training data:", X_train.shape, "Test data:", X_test.shape)
print(df.head(3))

Training data: (90, 7) Test data: (30, 7)
   Age_Score  Income_Score  Visit_Score  Random_Score  Engagement_Score  \
0  -1.534114      1.277677    -1.052060     -0.713276          0.746407   
1  -0.790474      0.471468     0.419777      1.065268          0.595331   
2   1.179297      0.067518     0.407684     -2.917161          2.257849   

   Constant  Age_Copy  Purchased  
0         1 -1.534114          0  
1         1 -0.790474          1  
2         1  1.179297          0  


## 1. What is Feature Selection?

### Definition
Feature selection means choosing the most useful input columns for a machine learning model. It keeps informative features and excludes columns that do not help prediction. It is different from feature extraction, which creates new features. Selection should be learned from training data, not the full dataset.

### Business Example
A shop records customer age, spending, visits, and an internal record number. The record number is only used to identify customers. The shop wants to predict whether a customer will buy again. It can select meaningful customer behavior columns instead of using every recorded field.

### AI/ML Use
Feature selection reduces unnecessary inputs before model training. It can make models faster and easier to understand. It may improve generalization when noisy features are removed. The selected columns must also be available when making future predictions.

### Python Example

In [3]:
print("Available features:", X_train.columns.tolist())
print("Number of features:", X_train.shape[1])

Available features: ['Age_Score', 'Income_Score', 'Visit_Score', 'Random_Score', 'Engagement_Score', 'Constant', 'Age_Copy']
Number of features: 7


### Explanation
The code lists the candidate input columns and counts them. These columns are potential predictors, not all guaranteed to be useful. We have not removed anything yet. The next topics show different ways to evaluate them.

## 2. Why Feature Selection?

### Definition
Feature selection helps reduce noise, computation, and model complexity. Too many inputs can increase training time. Some inputs repeat the same information or have little predictive value. Removing unnecessary columns may improve interpretability, but performance must be checked on unseen data.

### Business Example
An online store collects hundreds of customer attributes. Many fields are rarely used or repeat other information. Training on every field can waste resources. Selecting useful attributes makes the prediction system simpler to maintain.

### AI/ML Use
Feature selection can reduce overfitting, especially with small datasets. It helps distance-based models avoid noisy dimensions. Simpler models are often easier to explain to stakeholders. However, dropping an important feature can reduce accuracy.

### Python Example

In [5]:
print("Before selection:", X_train.shape[1], "features")
print("After excluding constant column:", X_train.drop(columns="Constant").shape[1], "features")

Before selection: 7 features
After excluding constant column: 6 features


### Explanation
The example shows a simple reduction in the number of columns. The constant column provides no variation. Removing it reduces an unnecessary input. Real selection should be evaluated using training and validation results.

## 3. Relevant Features

### Definition
Relevant features contain information useful for predicting the target. They may show a measurable relationship with the outcome. A feature can be useful alone or together with other features. Relevance should be assessed using training data and domain knowledge.

### Business Example
For customer purchase prediction, previous spending may be useful. Number of visits may also describe buying interest. These fields are available before the purchase decision. They can be relevant predictors when tested properly.

### AI/ML Use
Relevant inputs can help models distinguish classes. Their usefulness depends on the problem and algorithm. A weak individual relationship may still matter in combination. Use validation data to confirm whether a feature helps.

### Python Example

In [7]:
print("Candidate behavior features:", ["Income_Score", "Visit_Score", "Engagement_Score"])

Candidate behavior features: ['Income_Score', 'Visit_Score', 'Engagement_Score']


### Explanation
The code identifies three possible predictors for the sample task. This is a domain-based starting point rather than proof of importance. Statistical methods can help check relevance. Validation performance determines whether they are beneficial.

## 4. Irrelevant Features

### Definition
Irrelevant features do not provide useful information for predicting the target. They may be random, unrelated, or unavailable at prediction time. Such inputs can introduce noise. A feature that looks irrelevant alone might still interact with other variables.

### Business Example
A retailer stores a randomly assigned customer ID. The ID usually does not describe purchase behavior. Feeding it into a model can encourage accidental pattern learning. The business should prefer meaningful customer characteristics.

### AI/ML Use
Irrelevant inputs may increase model complexity. Some models can learn spurious relationships from them. Removing clearly unrelated fields can improve stability. Test the effect instead of assuming every weak feature is useless.

### Python Example

In [9]:
example = pd.DataFrame({"Customer_ID": [101, 102, 103], "Visits": [2, 5, 8]})
print(example.drop(columns="Customer_ID"))

   Visits
0       2
1       5
2       8


### Explanation
The example removes a customer identifier from the input table. The Visits column remains as a plausible predictor. An ID should generally be used for record matching rather than learning. The choice still depends on the meaning of the column.

## 5. Redundant Features

### Definition
Redundant features repeat information already represented by other inputs. Two columns may be duplicates or strongly related. Keeping both can complicate interpretation, especially in linear models. Correlation is one clue, but it does not identify every type of redundancy.

### Business Example
A company records annual salary and the same salary converted to monthly pay. Both describe nearly identical information. Keeping both may add little new value. One can often be removed after checking business requirements.

### AI/ML Use
Redundancy can increase multicollinearity in linear models. It can make coefficients unstable or harder to interpret. Tree-based models are often less sensitive, but importance can be split between related columns. Choose based on domain needs and validation.

### Python Example

In [11]:
print("Age vs Age_Copy correlation:", X_train["Age_Score"].corr(X_train["Age_Copy"]))

Age vs Age_Copy correlation: 1.0


### Explanation
The two columns are exact copies, so their correlation is 1.0. They carry the same information. One may be removed without losing that specific signal. This is an intentionally simple example of redundancy.

## 6. Correlation-Based Selection

### Definition
Correlation-based selection uses relationships between numerical columns. It can reveal highly related input features and potential redundancy. Correlation measures linear association, not causation. Correlation with the target alone does not capture every useful nonlinear pattern.

### Business Example
A bank records monthly income and yearly income. Their correlation is very high because one is derived from the other. The bank may keep only one for a simpler model. It must avoid removing useful information without testing.

### AI/ML Use
Correlation checks can reduce highly related predictors. They are especially useful when interpreting linear-model coefficients. Selection thresholds are not universal rules. Always compute correlations using training data only.

### Python Example

In [13]:
corr = X_train.drop(columns="Constant").corr().abs()
print("Correlation between copied features:", round(corr.loc["Age_Score", "Age_Copy"], 2))

Correlation between copied features: 1.0


### Explanation
The code calculates absolute correlations between training features. It highlights the exact-copy relationship. A high correlation suggests possible redundancy. It does not prove that either column should always be deleted.

## 7. Variance Threshold

### Definition
Variance Threshold removes features that change too little across observations. A constant column has zero variance. Such a column cannot distinguish different records. Low variance does not always mean low predictive value, particularly for rare but important events.

### Business Example
A store dataset has a Country column where every row is India. This column has no variation within that dataset. It cannot separate customers in that sample. The store can remove it for this model while keeping it in source records.

### AI/ML Use
VarianceThreshold is a simple filter method. It does not use the target labels. It can quickly remove constant or nearly constant columns. Set the threshold carefully because rare-event indicators may still matter.

### Python Example

In [15]:
selector = VarianceThreshold(threshold=0.0)
selector.fit(X_train)
print("Retained:", X_train.columns[selector.get_support()].tolist())

Retained: ['Age_Score', 'Income_Score', 'Visit_Score', 'Random_Score', 'Engagement_Score', 'Age_Copy']


### Explanation
The selector is fitted only on the training inputs. It detects the Constant column because its variance is zero. Other columns are retained. The same fitted selector can later transform test data.

## 8. Univariate Feature Selection

### Definition
Univariate selection evaluates each feature separately against the target. Statistical tests produce scores for individual predictors. SelectKBest keeps a chosen number of highest-scoring features. It can miss useful interactions between features.

### Business Example
A subscription company wants to predict cancellations. It measures how strongly each customer metric relates to cancellation. The highest-scoring metrics become a first shortlist. The company still tests the combined model before deciding.

### AI/ML Use
Univariate methods are fast and easy to apply. For classification with numerical inputs, an ANOVA F-test is one option. The scoring function should match the task and data. Fit the selector on training data to avoid leakage.

### Python Example

In [17]:
selector = SelectKBest(score_func=f_classif, k=3)
selector.fit(X_train.drop(columns="Constant"), y_train)
print("Selected:", X_train.drop(columns="Constant").columns[selector.get_support()].tolist())

Selected: ['Visit_Score', 'Random_Score', 'Engagement_Score']


### Explanation
The code scores features against the training labels. It keeps the top three by ANOVA F-score. The result is a statistical shortlist, not a guarantee of the best model. Test-set labels are not used during selection.

## 9. Mutual Information

### Definition
Mutual information measures how much knowing one variable tells us about another. It can capture some nonlinear relationships that correlation misses. A score near zero suggests little detected dependency. Estimates depend on sample size and settings.

### Business Example
A shopping app studies whether browsing patterns predict purchases. The relationship may not be a straight line. Mutual information can identify useful dependencies. The business can compare these features in a predictive model.

### AI/ML Use
Mutual information is a filter method for selecting predictors. It is useful when linear correlation is insufficient. It does not explain the direction of a relationship. Scores should be calculated on training data only.

### Python Example

In [19]:
scores = mutual_info_classif(X_train.drop(columns="Constant"), y_train, random_state=42)
print(pd.Series(scores, index=X_train.drop(columns="Constant").columns).round(3))

Age_Score           0.000
Income_Score        0.071
Visit_Score         0.378
Random_Score        0.144
Engagement_Score    0.233
Age_Copy            0.000
dtype: float64


### Explanation
The code estimates mutual information between each input and Purchased. Higher scores indicate more detected dependency in this sample. The numbers are estimates, not causal effects. They help rank candidates for further testing.

## 10. Recursive Feature Elimination

### Definition
Recursive Feature Elimination (RFE) repeatedly fits a model and removes lower-ranked features. It is a wrapper method because it uses an estimator during selection. The desired number of features must be chosen. RFE can be slower than simple filter methods.

### Business Example
A telecom company has several customer behavior indicators. It trains a model and gradually removes less useful ones. The process continues until a smaller set remains. The final set is checked against business knowledge and validation accuracy.

### AI/ML Use
RFE can account for the estimator used in selection. It works with models that expose coefficients or feature importance. Results depend on the chosen model. Perform RFE only within training data or cross-validation folds.

### Python Example

In [21]:
features = X_train.drop(columns=["Constant", "Age_Copy"])
rfe = RFE(LogisticRegression(max_iter=1000), n_features_to_select=3)
rfe.fit(features, y_train)
print("RFE selected:", features.columns[rfe.support_].tolist())

RFE selected: ['Visit_Score', 'Random_Score', 'Engagement_Score']


### Explanation
RFE trains logistic regression and removes lower-ranked inputs. It finishes with three selected features. The method is fitted using training rows only. The selected features should be evaluated on unseen observations.

## 11. Feature Importance

### Definition
Feature importance estimates how much each input contributes to a fitted model. Some models provide built-in importance scores. These scores depend on the model and can be biased or unstable. Importance does not establish that a feature causes the outcome.

### Business Example
A retailer builds a model to predict repeat purchases. It checks which customer attributes the model uses most. Managers can review whether the patterns make business sense. They should avoid treating an importance ranking as proof of cause.

### AI/ML Use
Tree-based models can provide feature importance values. These can guide feature selection and interpretation. Correlated features may share or distort importance. For stronger conclusions, compare with validation-based permutation importance.

### Python Example

In [23]:
features = X_train.drop(columns=["Constant", "Age_Copy"])
model = RandomForestClassifier(n_estimators=50, random_state=42)
model.fit(features, y_train)
print(pd.Series(model.feature_importances_, index=features.columns).sort_values(ascending=False).round(3))

Visit_Score         0.458
Engagement_Score    0.255
Random_Score        0.183
Age_Score           0.060
Income_Score        0.043
dtype: float64


### Explanation
The random forest is trained on the training subset. The output ranks the features by its built-in importance measure. Higher values mean greater contribution within this fitted forest. These values should be interpreted carefully and checked on validation data.